In [1]:
import numpy as np

import transpose_invariance as tpi

import skimage as ski
import skimage.registration as skr

See the analysis in `ai_output/gemini_flow_invariance.md`.

In [2]:
imgs = [ski.img_as_float(img)[:10, :64, :64]
        for img in tpi.get_3d_images()]
img = imgs[0]

In [3]:
img.shape

(10, 64, 64)

In [4]:
def f_tvl1(fixed, moving):
    return skr.optical_flow_tvl1(fixed, moving)

def f_ilk(fixed, moving):
    return skr.optical_flow_ilk(fixed, moving)

In [5]:
moving = np.roll(img, shift=(1, 2, 3), axis=(0, 1, 2))
ws_orig = f_tvl1(img, moving)

In [6]:
def rolled_proc_flow(img, axes, func):
    # Add small shift
    moving = np.roll(img, shift=(1, 2, 3), axis=(0, 1, 2))
    r_img = np.transpose(img, axes)
    r_moving = np.transpose(moving, axes)
    f_r_img = func(r_img, r_moving)
    # Output has shape (f_r_img.ndim,) + f_r_img.shape
    # Resort xes
    ax_order = np.argsort(axes)
    back_axes = [0] + list(ax_order + 1)
    # Resort axes and coord rows.
    return np.transpose(f_r_img, back_axes)[ax_order]

In [7]:
ws_rolled = rolled_proc_flow(img, (2, 1, 0), f_tvl1)
np.max(np.abs(ws_orig - ws_rolled))

np.float32(2.0116568e-07)

In [8]:
# All images are transpose invariant
for func in f_tvl1, f_ilk:
    print(func)
    for i, img in enumerate(imgs):
        moving = np.roll(img, shift=(1, 2, 3), axis=(0, 1, 2))
        orig = func(img, moving)
        print(f'Image {i}')
        for order in tpi.orderings:
            print(f'Ordering {order}')
            rolled = rolled_proc_flow(img, order, func)
            assert np.allclose(rolled, orig, atol=1e-5)

<function f_tvl1 at 0x10be0d440>
Image 0
Ordering (0, 2, 1)
Ordering (1, 2, 0)
Ordering (2, 1, 0)
Ordering (2, 0, 1)
Ordering (1, 0, 2)
Image 1
Ordering (0, 2, 1)
Ordering (1, 2, 0)
Ordering (2, 1, 0)
Ordering (2, 0, 1)
Ordering (1, 0, 2)
Image 2
Ordering (0, 2, 1)
Ordering (1, 2, 0)
Ordering (2, 1, 0)
Ordering (2, 0, 1)
Ordering (1, 0, 2)
<function f_ilk at 0x10be0c860>
Image 0
Ordering (0, 2, 1)
Ordering (1, 2, 0)
Ordering (2, 1, 0)
Ordering (2, 0, 1)
Ordering (1, 0, 2)
Image 1
Ordering (0, 2, 1)
Ordering (1, 2, 0)
Ordering (2, 1, 0)
Ordering (2, 0, 1)
Ordering (1, 0, 2)
Image 2
Ordering (0, 2, 1)
Ordering (1, 2, 0)
Ordering (2, 1, 0)
Ordering (2, 0, 1)
Ordering (1, 0, 2)
